# 1. Tải llama.cpp và Model từ Hugging Face
Chạy trên Colab Pro (khuyến nghị chọn A100 hoặc L4 GPU để chạy nhanh và mượt nhất)

In [ ]:
!pip install pyngrok --quiet
# Không dùng -U để tránh xung đột thư viện mặc định của Colab
!pip install huggingface_hub --quiet

# --- THAY ĐỔI MODEL Ở ĐÂY ---
HF_REPO = "mradermacher/Huihui-Qwen3.5-27B-Claude-4.6-Opus-abliterated-i1-GGUF"
MODEL_FILE = "Huihui-Qwen3.5-27B-Claude-4.6-Opus-abliterated.i1-Q4_K_M.gguf"
# ----------------------------

# Clone và build llama.cpp hỗ trợ CUDA bằng CMake (chuẩn mới)
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp && cmake -B build -DGGML_CUDA=ON && cmake --build build --config Release -j

# Tải model GGUF trực tiếp từ Hugging Face
!huggingface-cli download {HF_REPO} {MODEL_FILE} --local-dir . --local-dir-use-symlinks False


# 2. Khởi chạy Llama-server ở Background
Tối ưu hoá các tham số cho throughput trên GPU xịn (A100/L4)

In [ ]:
import os
import subprocess
import glob

LLAMA_ARG_PORT = "8000"
env = os.environ.copy()

# Tìm đường dẫn chính xác của llama-server sau khi CMake build (có thể ở build/bin/ hoặc build/)
server_path = glob.glob('./llama.cpp/build/**/llama-server', recursive=True)[0]

command = [
    server_path,
    "-m", MODEL_FILE,  # Dùng biến từ Cell trước
    "--port", LLAMA_ARG_PORT,
    
    # Tham số sinh text
    "--seed", "3407",
    "--temp", "0.6",
    "--top-p", "0.95",
    "--top-k", "20",
    "--min-p", "0.0",
    "--repeat-penalty", "1.0",
    
    # Tối ưu hoá phần cứng GPU
    "--flash-attn", "on",
    "--kv-unified",
    "--cache-type-k", "q8_0",
    "--cache-type-v", "q8_0",
    
    # Tối ưu cho Colab Pro (A100 40GB hoặc L4 24GB)
    "--batch-size", "2048",   # Logical batch size
    "--ubatch-size", "512",   # Physical batch size
    "--n_gpu_layers", "99",   # Offload toàn bộ
    "--ctx-size", "32768",    # Giữ context an toàn 32K
    
    # Server features
    "--parallel", "1",
    "--cont-batching",
    "--jinja",
    "--chat-template-kwargs", '{"enable_thinking":false}',
]

print(f"Đang khởi động llama-server tại {server_path} với model {MODEL_FILE}...")
llama_process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    env=env
)


# 3. Mở cổng qua Ngrok
Thêm `NGROK_TOKEN` vào tab **Secrets** (Biểu tượng chìa khoá ở thanh công cụ bên trái Colab).

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import time

try:
    ngrok_token = userdata.get("NGROK_TOKEN")
    !ngrok config add-authtoken {ngrok_token}
except Exception as e:
    print("LỖI: Vui lòng thêm NGROK_TOKEN vào menu Secrets của Colab!")

public_url = ngrok.connect(LLAMA_ARG_PORT)
print(f"\n👉 COPY ĐỊA CHỈ NÀY DÁN VÀO CẤU HÌNH API: {public_url}\n")

# Stream log của server ra output cell
for line in llama_process.stdout:
    print(line, end="")
